In [128]:
import os
from dotenv import load_dotenv
import uuid
import pymupdf
import chromadb
from langchain_core.documents import Document
import tqdm
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, SystemMessage


In [16]:
from sentence_transformers import SentenceTransformer

### PDF Reader

In [17]:
def read_pdfs(path = 'pdf'):
    all_files_in_path = os.listdir(path)
    all_pdfs = [file for file in all_files_in_path if file.lower().endswith('.pdf')]

    if not all_pdfs:
        return 'No PDF files found in the path'
    
    pdf_count = 0
    page_count = 0
    pdf_names = []
    all_documents : list[Document] = []

    for file in all_pdfs:
        file_path = os.path.join(path, file)
        all_page = pymupdf.open(file_path)

        pdf_count += 1
        page_count += len(all_page)
        pdf_names.append(file)

        for i, page in enumerate(all_page):
            text = page.get_text()
            if text.strip():
                doc = Document(
                page_content = text,
                metadata = {
                    'source': file,
                    "page": i,
                    'text_count': len(text)
                    }
                )
                all_documents.append(doc)
        all_page.close()
        

    print(f" ** Number of PDF found {pdf_count}\n ** {pdf_names}\n ** Total pages {page_count}")
    return all_documents

In [ ]:
documents = read_pdfs()

 ** Number of PDF found 3
 ** ['Attention is all you need.pdf', 'Prompt Engineering.pdf', "Retrieval-Augmented-Generation for LLM's.pdf"]
 ** Total pages 104


In [20]:
def split_documents(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    for chunk in chunks:
        chunk.metadata["text_count"] = len(chunk.page_content)
    return chunks

In [ ]:
pdf_chunks = split_documents(documents)

In [119]:
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print('Loading model', self.model_name)
        self.model = SentenceTransformer(self.model_name)

    def generate_embedding(self, data):
        docs = [content.page_content for content in data]
        embeddings = self.model.encode(
            docs,
            show_progress_bar=True
        )
        print("Shape", embeddings.shape)
        return embeddings
    

In [ ]:
class VectorStoreManager:
    def __init__(self, Persistent_path = 'vector_db', collection_name = 'pdf_database'):
        self.Persistent_path = Persistent_path
        self.collection_name = collection_name

        self.client = None
        self.collection = None

        self._initialize_connection()

    def _initialize_connection(self):
        self.client = chromadb.PersistentClient(
            self.Persistent_path
        )
        self.collection = self.client.get_or_create_collection(
            self.collection_name,
            metadata= {'description': 'This is a Vector Database for PDF files'}
            )
        
            
        
    def add_documents(self, documents, embedding_data):
        if self.collection.count() > 0:
            print(f"Collection already has {self.collection.count()} documents — skipping ingestion.")
            return

        texts =     [doc.page_content for doc in documents]
        metadatas = [dict(doc.metadata) for doc in documents]
        ids =       [f'id_{uuid.uuid4()}' for _ in documents]

        if len(embedding_data) != len(documents):
            raise ValueError("Number of embeddings and documents don't match")

        self.collection.add(
            documents = texts,
            ids = ids,
            metadatas=metadatas,
            embeddings=embedding_data.tolist()
        )
            


In [ ]:
class RAGRetriver:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def invoke(self, query, top_k = 5, similarity_threshold = 0.4):
        query_embedding  = self.embedding_manager.model.encode(query)

        result = self.vector_store.collection.query(
            query_embeddings = [query_embedding.tolist()],
            n_results=top_k,
        )

        if not result['documents'][0]:
            raise ValueError('No information is fetched form DB')
        
        all_docs = []
        for i, distance  in enumerate(result['distances'][0]):
            if distance  > similarity_threshold:
                all_docs.append(result['documents'][0][i])
        
        if not all_docs:
            raise ValueError("No documents passed the similarity threshold")

        llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0.6,                 
            max_tokens=None,                 
            timeout=None,
            max_retries=2
        )
        messages = [
            SystemMessage(content="You are a helpful Computer Science Rechercher who helps giving answers in a short note, and askes relative questions that the human might want to know"),
            HumanMessage(content= f"'query': {query}, 'given content': {all_docs}")
        ]

        response = llm.invoke(messages)


        return response.content


In [122]:
embedding_generator = EmbeddingManager()
embedding_data = embedding_generator.generate_embedding(pdf_chunks)

Loading model all-MiniLM-L6-v2


Batches: 100%|██████████| 18/18 [00:00<00:00, 19.03it/s]

Shape (551, 384)


In [123]:
db = VectorStoreManager()

Collection already has 2755 documents — skipping ingestion.


In [124]:
db.add_documents(pdf_chunks, embedding_data)

In [142]:
rag = RAGRetriver(embedding_generator, db)

In [143]:
result = rag.invoke('what is positional encoding')
print(result)

**Positional Encoding:**
Positional encoding is a technique used in transformer models to preserve the order of the sequence, as these models don't use recurrence or convolution. It involves adding positional information to the input embeddings, allowing the model to understand the relative or absolute position of tokens in the sequence.

**Key Points:**

* Added to input embeddings at the bottom of encoder and decoder stacks
* Same dimension as the model (d_model)
* Allows the model to understand the order of the sequence

**Would you like to know more about:**
1. How positional encoding is calculated?
2. The importance of positional encoding in transformer models?
3. Alternatives to positional encoding?
